# 💾 ChromaDB - Vector Database Fundamentals

**Store and search millions of embeddings efficiently**

---

## 📋 Overview

**What you'll learn:**
- What is a vector database
- Set up ChromaDB collections
- Store embeddings with metadata
- Perform similarity search
- Filter and query efficiently

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict
import uuid

print("✅ Setup complete")

## 📖 What is ChromaDB?

**Vector Database** = Database optimized for embeddings

**Why not regular database?**
```
Regular DB:  SELECT * WHERE name = 'exact match'
Vector DB:   SELECT * WHERE vector ~= 'similar meaning'
```

**ChromaDB Features:**
- ✅ Fast similarity search
- ✅ Metadata filtering
- ✅ Automatic embedding generation
- ✅ Persistent storage
- ✅ Easy to use (no setup!)

## 🏗️ Creating Your First Collection

In [ ]:
# Initialize ChromaDB client
client = chromadb.Client()  # In-memory (for testing)

# For persistent storage:
# client = chromadb.PersistentClient(path="./chroma_db")

# Create collection with embedding function
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="my_documents",
    embedding_function=embedding_fn,
    metadata={"description": "Sample document collection"}
)

print(f"✅ Created collection: {collection.name}")
print(f"   Items: {collection.count()}")

## 📥 Adding Documents

In [ ]:
# Add documents with metadata
documents = [
    "The weather is beautiful today",
    "It's a sunny day outside",
    "Python is a programming language",
    "JavaScript is used for web development",
    "Machine learning is a subset of AI",
]

# Generate unique IDs
ids = [str(uuid.uuid4()) for _ in documents]

# Metadata for each document
metadatas = [
    {"category": "weather", "source": "blog"},
    {"category": "weather", "source": "twitter"},
    {"category": "tech", "source": "docs"},
    {"category": "tech", "source": "docs"},
    {"category": "ai", "source": "article"},
]

# Add to collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ Added {collection.count()} documents")

## 🔍 Querying with Similarity Search

In [ ]:
# Query collection
results = collection.query(
    query_texts=["What's the climate like?"],
    n_results=3
)

print("Query: 'What's the climate like?'\n")
print("Top 3 Results:")
for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), 1):
    similarity = 1 - distance  # Convert distance to similarity
    print(f"\n{i}. Similarity: {similarity:.3f}")
    print(f"   Document: {doc}")
    print(f"   Metadata: {metadata}")

## 🎯 Filtering by Metadata

In [ ]:
# Query with metadata filter
results_filtered = collection.query(
    query_texts=["programming"],
    n_results=3,
    where={"category": "tech"}  # Only tech documents
)

print("Query: 'programming' (filtered: category=tech)\n")
for doc in results_filtered['documents'][0]:
    print(f"  • {doc}")

## 🔧 Production-Ready Vector Store Class

In [ ]:
class VectorStore:
    """Production-ready vector storage with ChromaDB."""
    
    def __init__(self, collection_name: str, persist_dir: str = None):
        if persist_dir:
            self.client = chromadb.PersistentClient(path=persist_dir)
        else:
            self.client = chromadb.Client()
        
        self.embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        
        # Get or create collection
        try:
            self.collection = self.client.create_collection(
                name=collection_name,
                embedding_function=self.embedding_fn
            )
        except:
            self.collection = self.client.get_collection(
                name=collection_name,
                embedding_function=self.embedding_fn
            )
    
    def add_texts(self, texts: List[str], metadatas: List[Dict] = None):
        """Add texts to collection."""
        ids = [str(uuid.uuid4()) for _ in texts]
        self.collection.add(
            documents=texts,
            metadatas=metadatas or [{} for _ in texts],
            ids=ids
        )
        return ids
    
    def search(self, query: str, n_results: int = 5, filter_dict: Dict = None):
        """Search for similar documents."""
        kwargs = {
            "query_texts": [query],
            "n_results": n_results
        }
        if filter_dict:
            kwargs["where"] = filter_dict
        
        return self.collection.query(**kwargs)
    
    def count(self) -> int:
        """Get number of documents."""
        return self.collection.count()
    
    def delete_collection(self):
        """Delete the entire collection."""
        self.client.delete_collection(self.collection.name)

# Example usage
store = VectorStore("products")

# Add products
products = [
    "Laptop 16GB RAM",
    "Wireless mouse",
    "USB keyboard"
]
store.add_texts(products)

# Search
results = store.search("computer accessories", n_results=2)
print(f"\nFound {len(results['documents'][0])} results:")
for doc in results['documents'][0]:
    print(f"  • {doc}")

# Clean up
store.delete_collection()

## ✅ Summary

### Key Takeaways:
- 💾 **ChromaDB**: Vector database made simple
- 🔍 **Similarity search**: Find related documents
- 🏷️ **Metadata**: Filter and organize
- ⚡ **Fast**: Optimized for vectors
- 💾 **Persistent**: Save to disk

### Common Operations:
```python
collection.add(documents, metadatas, ids)
collection.query(query_texts, n_results)
collection.get(ids)
collection.update(ids, documents)
collection.delete(ids)
```

### Next: `05_rag_systems/01_rag_fundamentals.ipynb`